[Reference](https://techwithram.medium.com/build-your-first-ai-agent-in-30-minutes-no-phd-required-353ddc612243$0)

In [1]:
import langchain
print(langchain.__version__)  # Should be 1.x.x

# Install if you haven't:
# pip install langchain langchain-core langchain-groq python-dotenv

1.3.6


# Your first model call

In [3]:
!pip install langchain langchain-core langchain-groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.1 MB/s eta 0:00:00


In [5]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
# Works with OpenAI, Anthropic, Groq - same interface
model = init_chat_model("groq:llama-3.1-8b-instant")
response = model.invoke("Explain what a REST API is in two sentences.")
print(response.content)

# Messages- the fundamental unit

In [6]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
# Build the conversation manually
conversation = [
    SystemMessage("You are a concise Python tutor. Keep answers short."),
    HumanMessage("What is a list comprehension?"),
]
response = model.invoke(conversation)
print(response.content)
# Continue the conversation - append both messages
conversation.append(response)
conversation.append(HumanMessage("Give me a quick example."))
followup = model.invoke(conversation)
print(followup.content)

# Tools — letting the model do things


In [7]:
from langchain.tools import tool

@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price for a given ticker symbol."""
    # In real code: call a finance API here
    prices = {"AAPL": "$192.45", "TSLA": "$248.10", "NVDA": "$875.30"}
    return prices.get(ticker.upper(), f"No price found for {ticker}")
@tool
def get_weather(city: str) -> str:
    """Get current weather conditions for a city."""
    return f"It's 22°C and partly cloudy in {city}."
# Bind the tools to the model
model_with_tools = model.bind_tools([get_stock_price, get_weather])

In [8]:
# Step 1: Send user question — model decides what to call
messages = [{"role": "user", "content": "What's the AAPL stock price?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute whatever tools the model requested
tools_map = {"get_stock_price": get_stock_price, "get_weather": get_weather}
for tool_call in ai_msg.tool_calls:
    print(f"Model wants: {tool_call['name']}({tool_call['args']})")
    result = tools_map[tool_call['name']].invoke(tool_call)
    messages.append(result)
# Step 3: Pass results back - model uses them in its final answer
final = model_with_tools.invoke(messages)
print(final.content)
# → "AAPL is currently trading at $192.45."

# Structured output — stop parsing strings


In [9]:
from pydantic import BaseModel, Field
from typing import List

class JobPosting(BaseModel):
    role: str = Field(description="Job title")
    company: str = Field(description="Company name")
    required_skills: List[str] = Field(description="List of required skills")
    remote: bool = Field(description="Whether the role is remote")
# Wrap the model - it will always return a JobPosting object
structured_model = model.with_structured_output(JobPosting)
job_text = """
Senior ML Engineer at DataCorp. Must know Python, PyTorch, and MLflow.
Fully remote position. Strong knowledge of distributed training preferred.
"""
posting = structured_model.invoke(job_text)
print(posting.role)            # → "Senior ML Engineer"
print(posting.required_skills)  # → ["Python", "PyTorch", "MLflow"]
print(posting.remote)           # → True

# Building an actual agent


In [10]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate

@tool
def search_docs(query: str) -> str:
    """Search internal documentation for answers."""
    # Swap this with a real vector search in production
    return f"Found 3 results for '{query}'. Top result: ..."
@tool
def get_user_info(user_id: str) -> str:
    """Look up account information for a user ID."""
    return f"User {user_id}: Plan=Pro, Joined=2024-01, Status=Active"
tools = [search_docs, get_user_info]
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful support agent. Use the tools available to answer user questions accurately."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])
agent = create_react_agent(model, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
result = executor.invoke({"input": "What plan is user U-9821 on?"})
print(result["output"])